# fucy_reboot — Hybrid RAG + CAG Pipeline

**Architecture:**
```
User Query
    │
    ▼
[QueryRewriter] → expands abbreviations, adds domain synonyms
    │
    ▼
[HybridRetriever] → BM25 keyword + dense vector search (RRF fusion)
    │
    ▼
[CrossEncoderReranker] → fine-grained (query, doc) pair scoring
    │
    ▼
[ContextAssembler] → relevance filter + MMR dedup + token budget
    │
    ▼
[FucyGenerator] → Gemini call (with optional CAG cache)
    │
    ▼
Validated JSON output
```

**CAG = Cache-Augmented Generation**: static domain data (`dataecu.json`, `annex.json`, ISO clauses) pre-loaded into Gemini's context cache — zero retrieval latency, reduced cost.


## 0. Install Dependencies

In [ ]:
# Install all required packages
!pip install -r requirements.txt

## 1. Configuration

In [ ]:
import sys
import os
from pathlib import Path

# Make sure fucy_reboot modules are importable
FUCY_DIR = Path(".").resolve()  # assumes notebook is run from fucy_reboot/
if str(FUCY_DIR) not in sys.path:
    sys.path.insert(0, str(FUCY_DIR))

import config

# ── USER SETTINGS ────────────────────────────────────────────────
# Set your Gemini API key here (or export GOOGLE_API_KEY before running)
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "YOUR_KEY_HERE")
config.GOOGLE_API_KEY = GOOGLE_API_KEY

# Path to your datasets folder (relative to this notebook)
DATASETS_DIR = "../rag_tara_gen/datasets"   # ← adjust if needed

# Force rebuild even if a cached index exists
FORCE_REBUILD = False

# Enable / disable CAG (Gemini context cache)
ENABLE_CAG = True
config.ENABLE_CAG = ENABLE_CAG

print(f"fucy_reboot dir : {FUCY_DIR}")
print(f"Datasets dir    : {Path(DATASETS_DIR).resolve()}")
print(f"Gemini model    : {config.GEMINI_MODEL}")
print(f"Embed model     : {config.EMBED_MODEL}")
print(f"CAG enabled     : {ENABLE_CAG}")
print(f"Chunk strategy  : {config.CHUNK_STRATEGY}")
print(f"Retrieval top-k : {config.RETRIEVAL_TOP_K}")
print(f"Reranker top-n  : {config.RERANKER_TOP_N}")

## 2. Data Ingestion

Loads all datasets from the `datasets/` folder:
- `dataecu.json` — ECU registry
- `annex.json` — domain annex
- `clauses/` — ISO/SAE 21434 clause files
- `atm.json`, `icsattack.json`, `mobileattack.json` — threat intelligence
- `reports_db/` — existing TARA reports
- `capec.xml`, `cwec.xml` — attack pattern / weakness databases

In [ ]:
from ingestion.ingestor import load_all_datasets

print("Loading datasets...")
records = load_all_datasets(DATASETS_DIR)
print(f"\nTotal records ingested: {len(records)}")

# Preview first record
import json
if records:
    sample = records[0]
    print("\nSample record:")
    print(f"  meta    : {sample['meta']}")
    print(f"  content : {sample['content'][:200]}...")

## 3. Semantic Chunking

Three strategies available:
| Strategy | Description |
|---|---|
| `sentence_window` | Groups N sentences with overlap (default) |
| `fixed_overlap` | Fixed character sliding window |
| `structure_aware` | Preserves JSON key groupings |

In [ ]:
from chunking.chunker import SemanticChunker

chunker = SemanticChunker(
    strategy=config.CHUNK_STRATEGY,
    chunk_size=config.CHUNK_SIZE,
    chunk_overlap=config.CHUNK_OVERLAP,
    sentences_per_chunk=config.SENTENCES_PER_CHUNK,
)

chunks = chunker.chunk(records)
print(f"Records : {len(records)}")
print(f"Chunks  : {len(chunks)}")
print(f"Avg chunks/record: {len(chunks)/max(len(records),1):.1f}")

# Preview
if chunks:
    c = chunks[0]
    print(f"\nSample chunk:")
    print(f"  meta    : {c['meta']}")
    print(f"  content : {c['content'][:300]}")

## 4. Embeddings

Uses `sentence-transformers/all-MiniLM-L6-v2` by default.  
Results cached to `cache/embeddings/` — subsequent runs are instant.

In [ ]:
from embeddings.embedder import BatchedEmbedder

embedder = BatchedEmbedder(
    model=config.EMBED_MODEL,
    batch_size=config.EMBED_BATCH_SIZE,
)

print("Embedding documents (uses disk cache if available)...")
embedded_docs = embedder.embed_from_records(chunks, use_cache=True)
print(f"\nEmbedded documents : {len(embedded_docs)}")
if embedded_docs and embedded_docs[0].embedding:
    print(f"Embedding dims     : {len(embedded_docs[0].embedding)}")

## 5. Document Index

Builds a Haystack `InMemoryDocumentStore` and persists it to `cache/index/`.  
Set `FORCE_REBUILD = True` in Cell 1 to force a fresh index.

In [ ]:
from indexing.index_manager import IndexManager

index_manager = IndexManager(cache_dir=config.INDEX_CACHE_DIR)

# Try loading from cache first
store = None
if not FORCE_REBUILD:
    store = index_manager.load()

if store is None:
    # Build fresh index from embedded docs
    print("Building index from embedded documents...")
    store = index_manager.build(embedded_docs)
    index_manager.save()

print(f"\nIndex contains {store.count_documents()} documents")

## 6. Hybrid Retriever (BM25 + Dense Vector + RRF)

In [ ]:
from retrieval.retriever import HybridRetriever

retriever = HybridRetriever(
    document_store=store,
    model=config.EMBED_MODEL,
    top_k=config.RETRIEVAL_TOP_K,
    bm25_weight=config.BM25_WEIGHT,
    vector_weight=config.VECTOR_WEIGHT,
)

# Quick smoke test
TEST_QUERY = "BMS battery management system damage scenarios"
test_docs = retriever.retrieve(TEST_QUERY)
print(f"Test query        : '{TEST_QUERY}'")
print(f"Retrieved docs    : {len(test_docs)}")
if test_docs:
    print(f"Top result source : {test_docs[0].meta.get('source', 'n/a')}")
    print(f"Top result score  : {test_docs[0].score:.4f}")

## 7. Cross-Encoder Re-Ranker

Uses `cross-encoder/ms-marco-MiniLM-L-6-v2` to score `(query, doc)` pairs and keep only the top-N most relevant results.

In [ ]:
from reranking.reranker import CrossEncoderReranker

reranker = CrossEncoderReranker(
    model=config.RERANKER_MODEL,
    top_n=config.RERANKER_TOP_N,
)

reranked_docs = reranker.rerank(TEST_QUERY, test_docs)
print(f"After reranking : {len(reranked_docs)} docs")
print("\nTop 5 results:")
for i, doc in enumerate(reranked_docs[:5]):
    source = doc.meta.get('source', 'n/a')
    score  = f"{doc.score:.4f}" if doc.score is not None else "n/a"
    print(f"  [{i+1}] score={score} source={source}")
    print(f"       {doc.content[:100]}...")

## 8. Context Assembly

Applies:
1. **Score filtering** — drops chunks below `MIN_RELEVANCE_SCORE`
2. **MMR deduplication** — removes near-duplicate chunks for diversity
3. **Token budget** — caps total context at `MAX_CONTEXT_CHARS`

In [ ]:
from context.assembler import ContextAssembler

assembler = ContextAssembler(
    min_score=config.MIN_RELEVANCE_SCORE,
    max_chars=config.MAX_CONTEXT_CHARS,
    enable_mmr=config.ENABLE_MMR_DEDUP,
    mmr_lambda=config.MMR_LAMBDA,
)

context_docs = assembler.assemble(reranked_docs, query=TEST_QUERY)
context_text = assembler.format_context(context_docs)

print(f"Context docs     : {len(context_docs)}")
print(f"Context chars    : {len(context_text):,}")
print("\n--- Context preview (first 800 chars) ---")
print(context_text[:800])

## 9. CAG — Cache-Augmented Generation (Optional)

Pre-loads static reference data into Gemini's context cache.  
Skip this cell (or set `ENABLE_CAG = False`) if you don't need it.

In [ ]:
cache_name = None

if config.ENABLE_CAG:
    from cag.cache_manager import CacheManager
    cache_manager = CacheManager(datasets_dir=DATASETS_DIR)
    cache_name = cache_manager.get_or_build()
    if cache_name:
        print(f"CAG active  : {cache_name}")
        print(f"Cache live  : {cache_manager.is_active}")
    else:
        print("CAG build failed — continuing without cache")
else:
    print("CAG disabled")

## 10. Generation — Single Query

Calls Gemini with assembled RAG context (+ optional CAG cache).  
Output is validated JSON; retries once with auto-repair on parse failure.

In [ ]:
from generation.generator import FucyGenerator

generator = FucyGenerator()

# Run generation on the test query
result = generator.generate(
    query=TEST_QUERY,
    context_docs=context_docs,
    context_text=context_text,
    cache_name=cache_name,
)

print(json.dumps(result, indent=2, ensure_ascii=False))

## 11. Full Pipeline — One-Liner Interface

The `FucyPipeline` orchestrates the entire flow automatically.

In [ ]:
from pipeline import FucyPipeline

# Build pipeline (reuses cached index if available)
pipeline = FucyPipeline(datasets_dir=DATASETS_DIR)

print("[1/2] Building index...")
doc_count = pipeline.build_index(force_rebuild=FORCE_REBUILD)
print(f"     Index ready: {doc_count} documents")

if config.ENABLE_CAG:
    print("[2/2] Setting up CAG cache...")
    cag = pipeline.build_cag_cache()
    print(f"     CAG: {cag or 'disabled/failed'}")

print("\nPipeline ready!")

## 12. Interactive Query Cell

Change `USER_QUERY` and re-run to query the pipeline.

In [ ]:
# ── CHANGE YOUR QUERY HERE ──────────────────────────────────────
USER_QUERY = "List all damage scenarios for the BMS with their SFOP ratings"
# ────────────────────────────────────────────────────────────────

result = pipeline.run(USER_QUERY)
print(json.dumps(result, indent=2, ensure_ascii=False))

## 13. Batch Queries

In [ ]:
BATCH_QUERIES = [
    "List all ECUs in the system with their asset types",
    "What are the top threat scenarios for CAN bus communication?",
    "Generate TARA damage scenarios for the Gateway ECU",
]

all_results = {}
for q in BATCH_QUERIES:
    print(f"\nQuery: {q}")
    result = pipeline.run(q)
    all_results[q] = result
    print(json.dumps(result, indent=2, ensure_ascii=False)[:400], "...")

print(f"\nTotal queries run: {len(all_results)}")

## 14. Save Results to JSON

In [ ]:
import time

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Save last single query result
ts = int(time.time())
out_path = output_dir / f"result_{ts}.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print(f"Saved to: {out_path}")

# Save batch results
batch_path = output_dir / f"batch_{ts}.json"
with open(batch_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f"Batch saved to: {batch_path}")

## 15. Evaluation Metrics

Measures retrieval quality: Precision@k, MRR, and source coverage.

In [ ]:
from evaluation.evaluator import evaluate_retrieval

# Define ground-truth relevant sources for a query
EVAL_QUERY = "BMS battery management system damage scenarios"
RELEVANT_SOURCES = {"dataecu", "tara_report", "annex"}

retrieved = pipeline.retrieve(EVAL_QUERY)

metrics = evaluate_retrieval(
    query=EVAL_QUERY,
    retrieved_docs=retrieved,
    relevant_sources=RELEVANT_SOURCES,
    k=config.EVAL_PRECISION_K,
)

print("Evaluation Metrics:")
for metric, value in metrics.items():
    print(f"  {metric:25s}: {value:.4f}")

## 16. Configuration Tuning

You can override any config value on a per-query basis using keyword arguments:

In [ ]:
# Run with custom config overrides (does NOT persist to config.py)
result_tuned = pipeline.run(
    "What are the cybersecurity goals for the BMS?",
    RETRIEVAL_TOP_K=50,      # retrieve more candidates
    RERANKER_TOP_N=15,       # keep more after reranking
    ENABLE_CAG=False,        # skip CAG for this query
)

print(json.dumps(result_tuned, indent=2, ensure_ascii=False))

## 17. CAG Cache Management

In [ ]:
if config.ENABLE_CAG:
    from cag.cache_manager import CacheManager
    cm = CacheManager(datasets_dir=DATASETS_DIR)

    print("Cache registry:")
    print(json.dumps(cm._registry, indent=2) if cm._registry else "  (empty)")
    print(f"Cache active  : {cm.is_active}")

    # Uncomment to refresh TTL:
    # cm.refresh()

    # Uncomment to force rebuild on next run:
    # cm.invalidate()
else:
    print("CAG is disabled")

## 18. Index Management

In [ ]:
from indexing.index_manager import IndexManager

idx = IndexManager(cache_dir=config.INDEX_CACHE_DIR)
store = idx.load()

if store:
    print(f"Loaded index    : {store.count_documents()} documents")

    # Retrieve sample documents by filter
    sample_docs = store.filter_documents(
        filters={"field": "meta.source", "operator": "==", "value": "dataecu"}
    )
    print(f"ECU docs found  : {len(sample_docs)}")
    if sample_docs:
        print(f"Sample ECU doc  : {sample_docs[0].content[:200]}")
else:
    print("No cached index found — run build step first")

---
## Module Reference

| Module | File | Purpose |
|---|---|---|
| Config | `config.py` | All tunable hyperparameters |
| Ingestion | `ingestion/ingestor.py` | JSON, XML, CAPEC loaders |
| Ingestion | `ingestion/normalizer.py` | Text cleaning & unicode normalization |
| Chunking | `chunking/chunker.py` | 3 strategies: sentence-window, fixed-overlap, structure-aware |
| Embeddings | `embeddings/embedder.py` | Batched embedding with disk cache |
| Indexing | `indexing/index_manager.py` | Persistent InMemoryDocumentStore |
| Retrieval | `retrieval/retriever.py` | Hybrid BM25 + vector with RRF |
| Retrieval | `retrieval/query_rewriter.py` | LLM query expansion |
| Retrieval | `retrieval/multi_query.py` | Multi-query variant retrieval |
| Re-ranking | `reranking/reranker.py` | Cross-encoder (ms-marco-MiniLM) |
| Context | `context/assembler.py` | MMR dedup, relevance filter, token budget |
| CAG | `cag/cache_builder.py` | Gemini context cache builder |
| CAG | `cag/cache_manager.py` | Cache lifecycle management |
| Generation | `generation/generator.py` | Gemini call + JSON validation + retry |
| Evaluation | `evaluation/evaluator.py` | Precision@k, MRR, coverage metrics |
| Pipeline | `pipeline.py` | Main orchestrator |
| CLI | `main.py` | Interactive CLI with save support |